
### Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [1]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
# Init model
model = init_chat_model("groq:qwen/qwen3-32b")

In [2]:
from pydantic import BaseModel,Field

# Init structured output for movie

# Tên Structure (Tên Class) 
# Khai báo các thông tin trả về như thuộc tính của Structure 
# Thông tin: tên thông tin, kiểu dữ liệu, mô tả (để LLM hiểu là phải trả về thông tin có đặc điểm như nào)

class Movie(BaseModel): 
    title:str = Field(description="Tiêu đề/tên của bộ phim") 
    year:int = Field(description="Năm bộ phim được xuất bản")
    director:str = Field(description="Đạo diễn của bộ phim")
    rating:float = Field(description="Điểm đánh giá của bộ phim") 

In [ ]:
# Result

RunnableBinding(bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001F5839C6660>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001F5839C7380>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'This year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The movies rating out of 10', 'type': 'number'}}, 'required': ['title', 'year', '

In [3]:
# Compare LLM with & without structured output
# Hãy cung cấp cho tôi thông tin về bộ phim Obsession
model.invoke("Hãy cung cấp cho tôi thông tin về bộ phim Obsession")

AIMessage(content='<think>\nOkay, the user is asking about the movie "Obsession." Let me start by recalling what I know about this film. I think it\'s a psychological thriller or maybe a horror movie. The title "Obsession" suggests themes related to fixation or intense desire. I need to verify the basic details like director, release year, cast, and plot.\n\nFirst, there might be multiple movies with the title "Obsession." I should check if the user is referring to a specific one. The most well-known one is probably "Obsession" (2016) directed by Ramin Bahrani, starring James Franco and Dakota Johnson. Another possibility is "Obsession" (1976), a TV movie by Robert Wise, which is a made-for-TV version of "The Three Faces of Eve," but that might be less likely. Alternatively, there\'s "The Obsession" (2015) about the history of Starbucks, but that\'s a documentary. I need to confirm which one the user is referring to.\n\nAssuming the user is talking about the 2016 film, let me outline t

In [4]:
model_with_structured_output = model.with_structured_output(Movie) 
model_with_structured_output.invoke("Hãy cung cấp cho tôi thông tin về bộ phim Obsession")

Movie(title='Obsession', year=2017, director='Daniele Lucchesi', rating=5.0)

In [ ]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)

response = model_with_structure.invoke("Provide details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking for details about the movie Inception. Let me check the tools available. There's a Movie function that requires title, year, director, and rating. I need to make sure I have all that information. Inception was directed by Christopher Nolan, released in 2010. The rating is probably around 8.8 on IMDb. Let me confirm the exact year and director. Yep, that's correct. The rating is 8.8. So I'll structure the tool call with those parameters. Make sure the JSON is correctly formatted with the required fields.\n", 'tool_calls': [{'id': 'r0n9zde78', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 170, 'prompt_tokens': 231, 'total_tokens': 401, 'completion_time': 0.272850748, 'completion_tokens_details': {'reasoning_tokens': 122}, 'prompt_time': 0

### Nested Structure

In [5]:
from pydantic import BaseModel, Field

# Init Actor (name, role)
class Actor(BaseModel): 
    name: str 
    role: str 

# Init MovieDetails (title, year, cast, genres, budget)
class MovieDetails(BaseModel): 
    title: str 
    year: int 
    cast: list[Actor] 
    genres: list[str] 
    budget: float | None 

In [7]:
movie_details_model = model.with_structured_output(MovieDetails) 
response = movie_details_model.invoke("Hãy cung cấp cho tôi thông tin về bộ phim Obsession")

In [16]:
cast_lst = [actor.name for actor in response.cast]
cast_lst 

['Michael Fassbender', 'Dakota Johnson', 'Shailene Woodley']

In [17]:
import os
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

In [18]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

In [19]:
agent = create_agent(
    model="gpt-5-mini", 
    response_format=ContactInfo 
)

In [23]:
result = agent.invoke({
    "messages": [
        {"role": "user", "content": "Extract contact info from Tom Cruise"}
    ]
})

result

{'messages': [HumanMessage(content='Extract contact info from Tom Cruise', additional_kwargs={}, response_metadata={}, id='cf742926-f78a-490b-9c23-4775d4d12323'),
  AIMessage(content='{"name":"Tom Cruise","email":"","phone":""}', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 601, 'prompt_tokens': 190, 'total_tokens': 791, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 576, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E1AK2pZ6n4TxiVL2Rs0sWhGG0MPhd', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f5b82-96d9-7693-813a-fe8e8dce4458-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 190, 'output_tokens': 601, 'total_tokens